# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic dataset info
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All entities are referenced and handled by their `@id` values.

In [ ]:
# List record sets and their fields
from pprint import pprint

record_sets = list(metadata.record_sets)
print(f"Record sets found ({len(record_sets)}):")
record_set_ids = []
for idx, rs in enumerate(record_sets):
    print(f"  {idx+1}. {rs['@id']}  --  label: {rs.get('name','')} ")
    record_set_ids.append(rs['@id'])
    # List fields
    print("     Fields:")
    for field in rs.get('fields', []):
        print(f"       - {field['@id']}: {field['name']} [{field.get('dataType','')}] (column: {field.get('column','')})")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

To keep things simple for demonstration, we use the first record set.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_set_ids:
    print(f'Loading records for record set: {record_set_id}')
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  -> Columns: {df.columns.tolist()}")
    print(f"  -> {len(df)} rows\n")

# Select one record set for demonstration
example_record_set_id = record_set_ids[0]
print(f"Preview of DataFrame for record set '{example_record_set_id}':")
display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section applies these steps to the first record set.

**Identify numeric fields:** Use the record set's fields metadata to determine what we can analyze.

In [ ]:
# Helper to map columns to fields and find numeric fields
def get_fields_by_type(rs, dtype='Float'):
    fields = rs.get('fields', [])
    numeric_fields = []
    for f in fields:
        dt = f.get('dataType', '')
        if isinstance(dt, list):
            dt = dt[0]
        if 'Float' in dt or 'Integer' in dt or dt.lower() in ['float', 'integer', 'number']:
            numeric_fields.append((f['@id'], f['name']))
    return numeric_fields

# Find numeric fields in the selected record set
selected_rs = None
for rs in metadata.record_sets:
    if rs['@id'] == example_record_set_id:
        selected_rs = rs
        break

numeric_fields = get_fields_by_type(selected_rs)
print("Numeric fields (by @id):")
pprint(numeric_fields)
print()

# Pick the first numeric field as example (e.g., for filtering and normalization)
if numeric_fields:
    numeric_field_id, numeric_field_label = numeric_fields[0]
    print(f"Using numeric field '{numeric_field_id}' for analysis.\n")
    # Prepare the DataFrame: ensure the column exists and convert to numeric
    df = dataframes[example_record_set_id].copy()
    col = numeric_field_id
    if col not in df.columns:
        print(f"Column '{col}' not in DataFrame columns: {df.columns.tolist()}")
    else:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        threshold = df[col].mean()  # Example: use mean as threshold

        filtered_df = df[df[col] > threshold]
        print(f"Filtered records with {col} > {threshold:.3f} (mean): {len(filtered_df)} records")
        display(filtered_df.head())

        # Normalize field
        filtered_df[f"{col}_normalized"] = (filtered_df[col] - filtered_df[col].mean()) / filtered_df[col].std()
        print(f"\nNormalized '{col}' for filtered records:")
        display(filtered_df[[col, f"{col}_normalized"]].head())

        # Try grouping by a categorical field, if any
        # Find first non-numeric field
        all_field_ids = [f['@id'] for f in selected_rs.get('fields', [])]
        cat_field_id = None
        for f in selected_rs.get('fields', []):
            dt = f.get('dataType', '')
            if (isinstance(dt, str) and dt not in ['Float', 'Integer']) and f['@id'] != numeric_field_id:
                cat_field_id = f['@id']
                break

        if cat_field_id and cat_field_id in filtered_df.columns:
            print(f"\nGrouped by '{cat_field_id}':")
            grouped_df = filtered_df.groupby(cat_field_id)[col].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
else:
    print("No numeric fields detected in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# We use the same df, numeric_field_id, and try to plot
if numeric_fields:
    col = numeric_field_id
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()

    # If we have a categorical field, try boxplot
    if cat_field_id and cat_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=cat_field_id, y=col, data=df)
        plt.title(f"{col} by {cat_field_id}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No numeric fields for visualization.")

## 6. Conclusion
In this notebook, you:
- Loaded dataset metadata and explored its structure using the `mlcroissant` library,
- Identified record sets and inspected structured fields by their `@id`,
- Extracted data as pandas DataFrames for analysis,
- Performed basic EDA such as filtering, normalization, and grouping on a numeric attribute using IDs,
- Created visualizations to understand data distributions and relationships between fields.

_For more information, see the original [dataset publication](https://sen.science/doi/10.71728/senscience.vq4a-28xa)._